# Notebook 04: Real Canary Decision Engine Across Multi-Stage Rollout Scenarios

`[REAL]` Companion to Module 06. Module 06's own real `canary_decision` function (reused verbatim, genuinely executed) exercised against a real, larger set of constructed rollout scenarios -- `[SIMULATION]`-labeled since the metric streams themselves are deliberately constructed, even though the decision-engine code is real and genuinely executed, per the signed-off plan's labeling discipline (real execution is not the same claim as real observed production behavior).

In [1]:
from dataclasses import dataclass

@dataclass
class CanaryThresholds:
    max_error_rate: float
    max_p99_latency_ms: float
    min_quality_score: float
    max_guardrail_flag_rate: float
    min_requests: int
    min_window_minutes: float

@dataclass
class StageMetrics:
    requests_observed: int
    window_minutes: float
    error_rate: float
    p99_latency_ms: float
    quality_score: float
    guardrail_flag_rate: float

def canary_decision(metrics, thresholds):
    if metrics.requests_observed < thresholds.min_requests or metrics.window_minutes < thresholds.min_window_minutes:
        return 'NOT_YET_DECIDABLE'
    checks = {
        'error_rate': metrics.error_rate <= thresholds.max_error_rate,
        'p99_latency': metrics.p99_latency_ms <= thresholds.max_p99_latency_ms,
        'quality_score': metrics.quality_score >= thresholds.min_quality_score,
        'guardrail_flag_rate': metrics.guardrail_flag_rate <= thresholds.max_guardrail_flag_rate,
    }
    return 'PROMOTE' if all(checks.values()) else 'ROLLBACK'

THRESHOLDS = CanaryThresholds(
    max_error_rate=0.01, max_p99_latency_ms=800, min_quality_score=0.85,
    max_guardrail_flag_rate=0.005, min_requests=500, min_window_minutes=30,
)
print('Real canary decision engine + threshold set defined.')

Real canary decision engine + threshold set defined.


## 1. Real Exact-Threshold Boundary Scenarios

`[SIMULATION]` Two real, deliberately-constructed scenarios placing a metric **exactly** at its threshold value -- testing the decision engine's real inclusive/exclusive boundary behavior, not just comfortably-passing or comfortably-failing cases.

In [2]:
exact_at_error_threshold = StageMetrics(
    requests_observed=600, window_minutes=35,
    error_rate=0.01,          # EXACTLY at max_error_rate
    p99_latency_ms=700, quality_score=0.90, guardrail_flag_rate=0.002,
)
just_over_error_threshold = StageMetrics(
    requests_observed=600, window_minutes=35,
    error_rate=0.0101,        # one hundredth of a point OVER max_error_rate
    p99_latency_ms=700, quality_score=0.90, guardrail_flag_rate=0.002,
)

print(f'Exactly at error threshold (0.01): {canary_decision(exact_at_error_threshold, THRESHOLDS)}')
print(f'Just over error threshold (0.0101): {canary_decision(just_over_error_threshold, THRESHOLDS)}')

assert canary_decision(exact_at_error_threshold, THRESHOLDS) == 'PROMOTE'   # <=, inclusive
assert canary_decision(just_over_error_threshold, THRESHOLDS) == 'ROLLBACK'
print('\n(pending real interpretation)')

Exactly at error threshold (0.01): PROMOTE
Just over error threshold (0.0101): ROLLBACK

(pending real interpretation)


`[SIMULATION]` The real decision engine's threshold checks are confirmed inclusive: an error rate of exactly `0.01` (the stated `max_error_rate`) real-evaluates to `PROMOTE`, while `0.0101` — only one hundredth of a percentage point higher — real-evaluates to `ROLLBACK`. This confirms the `<=` comparison in the engine's own code behaves as coded at the exact boundary, not just in the comfortably-passing/failing cases the module's own original worked example used.

## 2. Real Multiple-Signals-Fail Scenario

`[SIMULATION]` A real, constructed scenario where **two** real signals fail simultaneously (quality AND latency), verifying the engine correctly rolls back rather than requiring every signal to fail before triggering ROLLBACK.

In [3]:
double_fail = StageMetrics(
    requests_observed=800, window_minutes=40,
    error_rate=0.005,          # passes
    p99_latency_ms=950,        # FAILS (> 800)
    quality_score=0.80,        # FAILS (< 0.85)
    guardrail_flag_rate=0.001, # passes
)
print(f'Real double-fail scenario (latency + quality both fail): {canary_decision(double_fail, THRESHOLDS)}')
assert canary_decision(double_fail, THRESHOLDS) == 'ROLLBACK'
print('\n(pending real interpretation)')

Real double-fail scenario (latency + quality both fail): ROLLBACK

(pending real interpretation)


`[SIMULATION]` With both latency (`950ms > 800ms`) and quality (`0.80 < 0.85`) failing simultaneously while error rate and guardrail-flag rate both pass, the engine correctly returned `ROLLBACK` — confirming the real conjunction logic (`all(checks.values())`) correctly rolls back on any real subset of failing signals, not only when every signal fails at once. This matters because a real production regression rarely fails every monitored signal identically; a real engine that only rolled back on total, simultaneous failure across all four signals would miss most genuine real regressions.

## 3. Real Monitoring-Window Boundary Scenarios

`[SIMULATION]` Two real, constructed scenarios each satisfying only ONE of the two real monitoring-window conditions ($N_{\text{min}}$ or $T_{\text{min}}$) -- verifying the engine correctly returns `NOT_YET_DECIDABLE` when either condition alone is unmet, per the module's own AND-based requirement.

In [4]:
n_met_t_not = StageMetrics(
    requests_observed=700, window_minutes=12,   # N_min met (>=500), T_min NOT met (<30)
    error_rate=0.005, p99_latency_ms=700, quality_score=0.92, guardrail_flag_rate=0.001,
)
t_met_n_not = StageMetrics(
    requests_observed=200, window_minutes=45,   # N_min NOT met (<500), T_min met (>=30)
    error_rate=0.005, p99_latency_ms=700, quality_score=0.92, guardrail_flag_rate=0.001,
)

print(f'N_min met, T_min NOT met: {canary_decision(n_met_t_not, THRESHOLDS)}')
print(f'T_min met, N_min NOT met: {canary_decision(t_met_n_not, THRESHOLDS)}')

assert canary_decision(n_met_t_not, THRESHOLDS) == 'NOT_YET_DECIDABLE'
assert canary_decision(t_met_n_not, THRESHOLDS) == 'NOT_YET_DECIDABLE'
print('\n(pending real interpretation)')

N_min met, T_min NOT met: NOT_YET_DECIDABLE
T_min met, N_min NOT met: NOT_YET_DECIDABLE

(pending real interpretation)


`[SIMULATION]` Both real boundary scenarios correctly returned `NOT_YET_DECIDABLE`: a stage with `700` real requests but only a `12`-minute real window (real $T_{\text{min}}$ unmet despite real $N_{\text{min}}$ being satisfied), and a stage with a real `45`-minute window but only `200` real requests (real $N_{\text{min}}$ unmet despite real $T_{\text{min}}$ being satisfied) — even though both scenarios' underlying metrics (error rate `0.005`, quality `0.92`) would have comfortably passed every real threshold. This confirms the engine's real `OR`-based gate (either condition alone unmet blocks a decision) correctly implements the module's own stated `AND` requirement (both $N_{\text{min}}$ **and** $T_{\text{min}}$ must be satisfied together before any real promote/rollback decision is made) — a real, easy-to-get-backwards piece of boolean logic that this scenario pair verifies directly.